In [2]:
import os
import time
import zipfile
import urllib.request
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

np.random.seed(42)

In [3]:
url = "https://archive.ics.uci.edu/static/public/597/productivity+prediction+of+garment+employees.zip"
zip_path = "garment_productivity.zip"
extract_dir = "garment_dataset"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

csv_path = None

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_path = os.path.join(root, file)
            break
    if csv_path:
        break

if csv_path is None:
    raise FileNotFoundError("Dataset CSV file not found.")

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())
df.info()

print("\nMissing Values:")
display(df.isnull().sum())

Dataset loaded successfully!
Shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64

In [4]:
df = df.drop_duplicates().reset_index(drop=True)

df["department"] = df["department"].astype(str).str.strip()
df["department"] = df["department"].replace({"sweing": "sewing"})

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["meets_target"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)

print("Cleaned dataset shape:", df.shape)

print("\nClassification target distribution:")
display(df["meets_target"].value_counts())

Cleaned dataset shape: (1197, 16)

Classification target distribution:


meets_target
1    875
0    322
Name: count, dtype: int64

In [5]:
y_reg = df["actual_productivity"].astype(float)
y_clf = df["meets_target"].astype(int)

drop_cols = [
    "actual_productivity",
    "meets_target",
    "date"
]

X = df.drop(columns=drop_cols, errors="ignore").copy()

assert "actual_productivity" not in X.columns
assert "meets_target" not in X.columns

categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

print("Features:", X.columns.tolist())
print("Categorical features:", categorical_cols)
print("Numerical features:", numerical_cols)

Features: ['quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']
Categorical features: ['quarter', 'department', 'day']
Numerical features: ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


C:\Users\hasan pc\AppData\Local\Temp\ipykernel_22620\1963389376.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [6]:
indices = np.arange(len(X))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y_clf
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_reg_train = y_reg.iloc[train_idx].to_numpy()
y_reg_test = y_reg.iloc[test_idx].to_numpy()

y_clf_train = y_clf.iloc[train_idx].to_numpy()
y_clf_test = y_clf.iloc[test_idx].to_numpy()

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))

Training samples: 957
Testing samples: 240


In [7]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

X_train_skl = preprocessor.fit_transform(X_train)
X_test_skl = preprocessor.transform(X_test)

if hasattr(X_train_skl, "toarray"):
    X_train_skl = X_train_skl.toarray()

if hasattr(X_test_skl, "toarray"):
    X_test_skl = X_test_skl.toarray()

print("Processed training shape:", X_train_skl.shape)
print("Processed testing shape:", X_test_skl.shape)

Processed training shape: (957, 23)
Processed testing shape: (240, 23)


In [8]:
linear_model = LinearRegression()

start_train = time.perf_counter()
linear_model.fit(X_train_skl, y_reg_train)
sk_linear_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()
sk_reg_predictions = linear_model.predict(X_test_skl)
sk_linear_pred_time = time.perf_counter() - start_pred

sk_reg_metrics = {
    "MAE": mean_absolute_error(y_reg_test, sk_reg_predictions),
    "RMSE": np.sqrt(mean_squared_error(y_reg_test, sk_reg_predictions)),
    "R2": r2_score(y_reg_test, sk_reg_predictions)
}

print("SCIKIT-LEARN LINEAR REGRESSION")
print(sk_reg_metrics)
print("Training Time:", sk_linear_train_time)
print("Prediction Time:", sk_linear_pred_time)

SCIKIT-LEARN LINEAR REGRESSION
{'MAE': 0.10850276488125464, 'RMSE': np.float64(0.14575409595200192), 'R2': 0.2684461883259477}
Training Time: 0.018825000006472692
Prediction Time: 0.0014993000077083707


In [9]:
logistic_model = LogisticRegression(
    max_iter=3000,
    C=1.0,
    solver="lbfgs",
    random_state=42
)

start_train = time.perf_counter()
logistic_model.fit(X_train_skl, y_clf_train)
sk_log_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()
sk_clf_predictions = logistic_model.predict(X_test_skl)
sk_log_pred_time = time.perf_counter() - start_pred

sk_clf_metrics = {
    "Accuracy": accuracy_score(y_clf_test, sk_clf_predictions),
    "Precision": precision_score(
        y_clf_test, sk_clf_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_clf_test, sk_clf_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_clf_test, sk_clf_predictions, zero_division=0
    )
}

print("SCIKIT-LEARN LOGISTIC REGRESSION")
print(sk_clf_metrics)
print("Training Time:", sk_log_train_time)
print("Prediction Time:", sk_log_pred_time)

SCIKIT-LEARN LOGISTIC REGRESSION
{'Accuracy': 0.7083333333333334, 'Precision': 0.7536231884057971, 'Recall': 0.8914285714285715, 'F1': 0.8167539267015707}
Training Time: 0.04388949999702163
Prediction Time: 0.000713700006599538


In [10]:
X_train_num = X_train[numerical_cols].copy()
X_test_num = X_test[numerical_cols].copy()

X_train_cat = X_train[categorical_cols].copy()
X_test_cat = X_test[categorical_cols].copy()

for col in numerical_cols:
    X_train_num[col] = pd.to_numeric(
        X_train_num[col], errors="coerce"
    )
    X_test_num[col] = pd.to_numeric(
        X_test_num[col], errors="coerce"
    )

num_medians = X_train_num.median()

X_train_num = X_train_num.fillna(num_medians)
X_test_num = X_test_num.fillna(num_medians)

cat_modes = {}

for col in categorical_cols:
    mode_values = X_train_cat[col].mode(dropna=True)

    fill_value = (
        mode_values.iloc[0]
        if not mode_values.empty
        else "Unknown"
    )

    cat_modes[col] = fill_value

    X_train_cat[col] = X_train_cat[col].fillna(fill_value)
    X_test_cat[col] = X_test_cat[col].fillna(fill_value)

X_train_cat = X_train_cat.astype(str)
X_test_cat = X_test_cat.astype(str)

X_train_cat_encoded = pd.get_dummies(
    X_train_cat,
    columns=categorical_cols,
    dtype=float
)

X_test_cat_encoded = pd.get_dummies(
    X_test_cat,
    columns=categorical_cols,
    dtype=float
)

X_test_cat_encoded = X_test_cat_encoded.reindex(
    columns=X_train_cat_encoded.columns,
    fill_value=0
)

num_means = X_train_num.mean()
num_stds = X_train_num.std(ddof=0).replace(0, 1)

X_train_num_scaled = (
    X_train_num - num_means
) / num_stds

X_test_num_scaled = (
    X_test_num - num_means
) / num_stds

X_train_manual = np.hstack([
    X_train_num_scaled.to_numpy(dtype=float),
    X_train_cat_encoded.to_numpy(dtype=float)
])

X_test_manual = np.hstack([
    X_test_num_scaled.to_numpy(dtype=float),
    X_test_cat_encoded.to_numpy(dtype=float)
])

X_train_manual = np.column_stack([
    np.ones(len(X_train_manual)),
    X_train_manual
])

X_test_manual = np.column_stack([
    np.ones(len(X_test_manual)),
    X_test_manual
])

print("Manual training shape:", X_train_manual.shape)
print("Manual testing shape:", X_test_manual.shape)


Manual training shape: (957, 24)
Manual testing shape: (240, 24)


In [11]:
def manual_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


def manual_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

In [12]:
start_train = time.perf_counter()

manual_linear_weights = np.linalg.lstsq(
    X_train_manual,
    y_reg_train,
    rcond=None
)[0]

manual_linear_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()

manual_reg_predictions = (
    X_test_manual @ manual_linear_weights
)

manual_linear_pred_time = time.perf_counter() - start_pred

manual_reg_metrics = manual_regression_metrics(
    y_reg_test,
    manual_reg_predictions
)

print("MANUAL LINEAR REGRESSION")
print(manual_reg_metrics)
print("Training Time:", manual_linear_train_time)
print("Prediction Time:", manual_linear_pred_time)

MANUAL LINEAR REGRESSION
{'MAE': np.float64(0.10850276488125479), 'RMSE': np.float64(0.14575409595200187), 'R2': np.float64(0.26844618832594813)}
Training Time: 0.011662599994451739
Prediction Time: 0.00023890000011306256


In [13]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def manual_logistic_train(
    X,
    y,
    learning_rate=0.01,
    epochs=5000,
    l2=0.0,
    tolerance=1e-8
):
    n_samples, n_features = X.shape
    weights = np.zeros(n_features, dtype=float)

    for epoch in range(epochs):
        probabilities = sigmoid(X @ weights)
        errors = probabilities - y

        gradient = (X.T @ errors) / n_samples
        gradient[1:] += l2 * weights[1:]

        new_weights = weights - learning_rate * gradient

        if np.linalg.norm(new_weights - weights) < tolerance:
            weights = new_weights
            break

        weights = new_weights

    return weights


def manual_logistic_predict_proba(X, weights):
    return sigmoid(X @ weights)


def manual_logistic_predict(X, weights, threshold=0.5):
    probabilities = manual_logistic_predict_proba(X, weights)
    return (probabilities >= threshold).astype(int)


start_train = time.perf_counter()

manual_logistic_weights = manual_logistic_train(
    X_train_manual,
    y_clf_train,
    learning_rate=0.01,
    epochs=5000,
    l2=0.0
)

manual_log_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()

manual_clf_predictions = manual_logistic_predict(
    X_test_manual,
    manual_logistic_weights
)

manual_log_pred_time = time.perf_counter() - start_pred

manual_clf_metrics = manual_classification_metrics(
    y_clf_test,
    manual_clf_predictions
)

print("MANUAL LOGISTIC REGRESSION")
print(manual_clf_metrics)
print("Training Time:", manual_log_train_time)
print("Prediction Time:", manual_log_pred_time)

MANUAL LOGISTIC REGRESSION
{'Accuracy': np.float64(0.7125), 'Precision': np.float64(0.7431192660550459), 'Recall': np.float64(0.9257142857142857), 'F1': np.float64(0.8244274809160306)}
Training Time: 0.3877895999903558
Prediction Time: 0.00034359999699518085


In [14]:
start_train = time.perf_counter()

optimized_logistic_weights = manual_logistic_train(
    X_train_manual,
    y_clf_train,
    learning_rate=0.05,
    epochs=10000,
    l2=0.01,
    tolerance=1e-7
)

optimized_log_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()

optimized_clf_predictions = manual_logistic_predict(
    X_test_manual,
    optimized_logistic_weights,
    threshold=0.5
)

optimized_log_pred_time = time.perf_counter() - start_pred

optimized_clf_metrics = manual_classification_metrics(
    y_clf_test,
    optimized_clf_predictions
)

print("OPTIMIZED MANUAL LOGISTIC REGRESSION")
print(optimized_clf_metrics)
print("Training Time:", optimized_log_train_time)
print("Prediction Time:", optimized_log_pred_time)


OPTIMIZED MANUAL LOGISTIC REGRESSION
{'Accuracy': np.float64(0.7125), 'Precision': np.float64(0.740909090909091), 'Recall': np.float64(0.9314285714285714), 'F1': np.float64(0.8253164556962026)}
Training Time: 0.5274767999944743
Prediction Time: 0.00011470000026747584


In [15]:
def manual_ridge_regression(X, y, alpha=0.01):
    n_features = X.shape[1]

    identity = np.eye(n_features)
    identity[0, 0] = 0.0

    weights = np.linalg.solve(
        X.T @ X + alpha * identity,
        X.T @ y
    )

    return weights


start_train = time.perf_counter()

optimized_linear_weights = manual_ridge_regression(
    X_train_manual,
    y_reg_train,
    alpha=0.01
)

optimized_linear_train_time = time.perf_counter() - start_train

start_pred = time.perf_counter()

optimized_reg_predictions = (
    X_test_manual @ optimized_linear_weights
)

optimized_linear_pred_time = time.perf_counter() - start_pred

optimized_reg_metrics = manual_regression_metrics(
    y_reg_test,
    optimized_reg_predictions
)

print("OPTIMIZED MANUAL RIDGE REGRESSION")
print(optimized_reg_metrics)
print("Training Time:", optimized_linear_train_time)
print("Prediction Time:", optimized_linear_pred_time)

OPTIMIZED MANUAL RIDGE REGRESSION
{'MAE': np.float64(0.1085016286798471), 'RMSE': np.float64(0.1457537916981873), 'R2': np.float64(0.26844924248101665)}
Training Time: 0.005283599995891564
Prediction Time: 0.00013690000923816115


In [16]:
regression_comparison = pd.DataFrame([
    {
        "Implementation": "Scikit-learn Linear Regression",
        **sk_reg_metrics,
        "Training Time (s)": sk_linear_train_time,
        "Prediction Time (s)": sk_linear_pred_time
    },
    {
        "Implementation": "Manual Linear Regression",
        **manual_reg_metrics,
        "Training Time (s)": manual_linear_train_time,
        "Prediction Time (s)": manual_linear_pred_time
    },
    {
        "Implementation": "Optimized Manual Ridge Regression",
        **optimized_reg_metrics,
        "Training Time (s)": optimized_linear_train_time,
        "Prediction Time (s)": optimized_linear_pred_time
    }
])

print("REGRESSION COMPARISON")
display(regression_comparison.round(6))

REGRESSION COMPARISON


,Implementation,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Scikit-learn Linear Regression,0.108503,0.145754,0.268446,0.018825,0.001499
1,Manual Linear Regression,0.108503,0.145754,0.268446,0.011663,0.000239
2,Optimized Manual Ridge Regression,0.108502,0.145754,0.268449,0.005284,0.000137


In [17]:
classification_comparison = pd.DataFrame([
    {
        "Implementation": "Scikit-learn Logistic Regression",
        **sk_clf_metrics,
        "Training Time (s)": sk_log_train_time,
        "Prediction Time (s)": sk_log_pred_time
    },
    {
        "Implementation": "Manual Logistic Regression",
        **manual_clf_metrics,
        "Training Time (s)": manual_log_train_time,
        "Prediction Time (s)": manual_log_pred_time
    },
    {
        "Implementation": "Optimized Manual Logistic Regression",
        **optimized_clf_metrics,
        "Training Time (s)": optimized_log_train_time,
        "Prediction Time (s)": optimized_log_pred_time
    }
])

print("CLASSIFICATION COMPARISON")
display(classification_comparison.round(6))

CLASSIFICATION COMPARISON


,Implementation,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Scikit-learn Logistic Regression,0.708333,0.753623,0.891429,0.816754,0.043889,0.000714
1,Manual Logistic Regression,0.712500,0.743119,0.925714,0.824427,0.387790,0.000344
2,Optimized Manual Logistic Regression,0.712500,0.740909,0.931429,0.825316,0.527477,0.000115


In [18]:
reg_table = regression_comparison.copy()
reg_table["Task"] = "Regression"

clf_table = classification_comparison.copy()
clf_table["Task"] = "Classification"

combined_comparison = pd.concat(
    [reg_table, clf_table],
    ignore_index=True,
    sort=False
)

print("COMBINED MODEL COMPARISON")
display(combined_comparison.round(6))

COMBINED MODEL COMPARISON


,Implementation,MAE,RMSE,R2,Training Time (s),Prediction Time (s),Task,Accuracy,Precision,Recall,F1
0,Scikit-learn Linear Regression,0.108503,0.145754,0.268446,0.018825,0.001499,Regression,NaN,NaN,NaN,NaN
1,Manual Linear Regression,0.108503,0.145754,0.268446,0.011663,0.000239,Regression,NaN,NaN,NaN,NaN
2,Optimized Manual Ridge Regression,0.108502,0.145754,0.268449,0.005284,0.000137,Regression,NaN,NaN,NaN,NaN
3,Scikit-learn Logistic Regression,NaN,NaN,NaN,0.043889,0.000714,Classification,0.708333,0.753623,0.891429,0.816754
4,Manual Logistic Regression,NaN,NaN,NaN,0.387790,0.000344,Classification,0.712500,0.743119,0.925714,0.824427
5,Optimized Manual Logistic Regression,NaN,NaN,NaN,0.527477,0.000115,Classification,0.712500,0.740909,0.931429,0.825316


In [19]:
regression_comparison.to_csv(
    "regression_comparison.csv",
    index=False
)

classification_comparison.to_csv(
    "classification_comparison.csv",
    index=False
)

combined_comparison.to_csv(
    "combined_comparison.csv",
    index=False
)

print("Comparison tables saved successfully.")

Comparison tables saved successfully.


In [20]:
print("=" * 60)
print("FINAL PERFORMANCE SUMMARY")
print("=" * 60)

print("\nREGRESSION:")
print("Scikit-learn RMSE:", round(sk_reg_metrics["RMSE"], 6))
print("Manual RMSE:", round(manual_reg_metrics["RMSE"], 6))
print("Optimized Manual RMSE:", round(optimized_reg_metrics["RMSE"], 6))

print("\nCLASSIFICATION:")
print("Scikit-learn Accuracy:", round(sk_clf_metrics["Accuracy"], 6))
print("Manual Accuracy:", round(manual_clf_metrics["Accuracy"], 6))
print("Optimized Manual Accuracy:", round(optimized_clf_metrics["Accuracy"], 6))

print("\nRUNTIME:")
print("Scikit-learn Linear Regression:", round(sk_linear_train_time, 6), "seconds")
print("Manual Linear Regression:", round(manual_linear_train_time, 6), "seconds")
print("Scikit-learn Logistic Regression:", round(sk_log_train_time, 6), "seconds")
print("Manual Logistic Regression:", round(manual_log_train_time, 6), "seconds")

FINAL PERFORMANCE SUMMARY

REGRESSION:
Scikit-learn RMSE: 0.145754
Manual RMSE: 0.145754
Optimized Manual RMSE: 0.145754

CLASSIFICATION:
Scikit-learn Accuracy: 0.708333
Manual Accuracy: 0.7125
Optimized Manual Accuracy: 0.7125

RUNTIME:
Scikit-learn Linear Regression: 0.018825 seconds
Manual Linear Regression: 0.011663 seconds
Scikit-learn Logistic Regression: 0.043889 seconds
Manual Logistic Regression: 0.38779 seconds


In [22]:
print("KEY OBSERVATIONS")
print("-" * 60)

print("1. Regression models were evaluated using MAE, RMSE, and R2.")
print("2. Classification models were evaluated using accuracy, precision, recall, and F1-score.")
print("3. All models use the same fixed train-test split.")
print("4. Actual productivity was excluded from classification features.")
print("5. Manual preprocessing and model training were implemented using NumPy and Pandas.")
print("6. Optimized manual models use regularization and vectorized numerical operations.")
print("7. Runtime differences depend on optimization methods, matrix sizes, and implementation overhead.")
print("8. Ridge regularization and logistic regression tuning can change predictive performance.")



KEY OBSERVATIONS
------------------------------------------------------------
1. Regression models were evaluated using MAE, RMSE, and R2.
2. Classification models were evaluated using accuracy, precision, recall, and F1-score.
3. All models use the same fixed train-test split.
4. Actual productivity was excluded from classification features.
5. Manual preprocessing and model training were implemented using NumPy and Pandas.
6. Optimized manual models use regularization and vectorized numerical operations.
7. Runtime differences depend on optimization methods, matrix sizes, and implementation overhead.
8. Ridge regularization and logistic regression tuning can change predictive performance.
